# AgentsVille AI Trip Planner

An AI-powered travel planning system that generates and refines vacation itineraries using:
- **Large Language Models (LLMs)** via the OpenAI API
- **Structured data validation** with Pydantic models
- **Tool-based reasoning** with the ReAct (Reasoning + Acting) framework

## Workflow
1. Define traveler preferences (`VacationInfo`)
2. Gather weather forecasts and available activities (simulated APIs)
3. Generate an initial itinerary (`ItineraryAgent`)
4. Evaluate the itinerary against several automated checks
5. Revise the itinerary with the `ItineraryRevisionAgent` (ReAct loop)
6. Produce a final narrative trip summary

## 1. Setup

Install dependencies and configure the OpenAI client.

In [ ]:
# Install dependencies (run once)
# !pip install -r requirements.txt

In [ ]:
import json
import os
from datetime import date

from openai import OpenAI

from project_lib import (
    VacationInfo,
    TravelPlan,
    get_weather_forecast,
    get_available_activities,
    run_all_evaluations,
    print_eval_results,
    ItineraryAgent,
    ItineraryRevisionAgent,
    generate_trip_summary,
    save_itinerary,
)

# Set your OpenAI API key before running.
# Option 1: environment variable (recommended)
#   export OPENAI_API_KEY="sk-..."
# Option 2: hard-code here (not recommended for production)
#   os.environ["OPENAI_API_KEY"] = "sk-..."

client = OpenAI()  # reads OPENAI_API_KEY from environment
MODEL = "gpt-4o"

print("Setup complete.")

## 2. Define Traveler Preferences

We use the `VacationInfo` Pydantic model to capture and validate the traveler's details.

In [ ]:
vacation_info = VacationInfo(
    destination="AgentsVille",
    start_date=date(2026, 6, 10),
    end_date=date(2026, 6, 14),
    interests=["history", "food", "outdoor activities", "art"],
    budget=400.0,
    constraints=["no extreme sports"],
)

print("Vacation Info:")
print(vacation_info.model_dump_json(indent=2))

## 3. Gather Data

Retrieve the weather forecast and available activities for each day of the trip.
In a real application these would call external travel and weather APIs.

In [ ]:
start_str = str(vacation_info.start_date)
end_str = str(vacation_info.end_date)

# Weather forecast for the entire trip
weather_forecast = get_weather_forecast(
    vacation_info.destination, start_str, end_str
)

# Available activities for each day
activities_by_date = {}
from datetime import timedelta

current = vacation_info.start_date
while current <= vacation_info.end_date:
    date_str = str(current)
    activities_by_date[date_str] = get_available_activities(
        vacation_info.destination, date_str
    )
    current += timedelta(days=1)

print("Weather Forecast:")
print(json.dumps(weather_forecast, indent=2))

print("\nAvailable Activities by Date:")
print(json.dumps(activities_by_date, indent=2))

## 4. Generate the Initial Itinerary

The `ItineraryAgent` sends the traveler info, weather, and activities to the LLM
and receives a structured `TravelPlan` back (enforced by Pydantic's `response_format`).

In [ ]:
itinerary_agent = ItineraryAgent(client=client, model=MODEL)

print("Generating initial itinerary...")
travel_plan = itinerary_agent.generate(
    vacation_info=vacation_info,
    weather_forecast=weather_forecast,
    activities_by_date=activities_by_date,
)

print("\nInitial Travel Plan:")
print(travel_plan.model_dump_json(indent=2))

## 5. Evaluate the Itinerary

Run automated checks to verify:
- **Weather compatibility** — no outdoor activities on rainy days
- **Activity availability** — all activities exist on their scheduled dates
- **Budget accuracy** — total cost is correctly summed and within budget
- **Minimum activities** — at least 2 activities per day
- **City / date correctness** — all dates fall within the travel window

In [ ]:
eval_results = run_all_evaluations(
    travel_plan=travel_plan,
    vacation_info=vacation_info,
    weather_forecast=weather_forecast,
)

print_eval_results(eval_results)

all_passed = all(r["passed"] for r in eval_results)
print(f"All checks passed: {all_passed}")

## 6. Revise the Itinerary (ReAct Loop)

If any checks failed, the `ItineraryRevisionAgent` uses the ReAct framework to
iteratively fix issues:

```
THOUGHT  → What needs to be fixed?
ACTION   → Call a tool (get_activities_by_date_tool, calculator_tool, run_evals_tool …)
OBSERVATION → Review the result and plan the next step
```

The loop terminates when `final_answer_tool` is called with a plan that passes all checks.

In [ ]:
if all_passed:
    print("All checks already passed — no revision needed.")
    final_plan = travel_plan
else:
    revision_agent = ItineraryRevisionAgent(client=client, model=MODEL, max_iterations=10)
    final_plan = revision_agent.revise(
        travel_plan=travel_plan,
        vacation_info=vacation_info,
        weather_forecast=weather_forecast,
        eval_results=eval_results,
    )

print("\nFinal Travel Plan:")
print(final_plan.model_dump_json(indent=2))

### 6a. Verify the Revised Plan

Run evaluations one more time to confirm all issues have been resolved.

In [ ]:
final_eval_results = run_all_evaluations(
    travel_plan=final_plan,
    vacation_info=vacation_info,
    weather_forecast=weather_forecast,
)

print_eval_results(final_eval_results)
print(f"All checks passed: {all(r['passed'] for r in final_eval_results)}")

## 7. Generate a Trip Summary

Ask the LLM to write a short narrative description of the final itinerary.

In [ ]:
summary = generate_trip_summary(
    client=client,
    travel_plan=final_plan,
    vacation_info=vacation_info,
    model=MODEL,
)

print("Trip Summary")
print("=" * 60)
print(summary)

## 8. Save the Itinerary

Persist the final plan, traveler info, and summary to the `outputs/` directory.

In [ ]:
output_path = save_itinerary(
    travel_plan=final_plan,
    vacation_info=vacation_info,
    summary=summary,
)

# Preview the saved file
with open(output_path) as fh:
    saved = json.load(fh)

print(json.dumps(saved, indent=2))